# Mesh generation - Gridgen quadtree

Build a **quadtree** unstructured (DISV) grid with the Gridgen program: start from a coarse base grid and recursively quarter cells near the area of interest and along the river network.

Part of the **mesh-generation** series, adapted from the FloPy watershed geoprocessing example (Hughes and others, 2023, *FloPy Workflows for Creating Structured and Unstructured MODFLOW Models*, Groundwater, https://doi.org/10.1111/gwat.13327). Each notebook builds one family of grids over the same synthetic watershed, samples a fine topographic raster onto the grid, and intersects the river network with the grid cells.

- **Rectilinear (DIS + LGR)** (`mesh-generation-rectilinear`) - structured grids: constant and variable spacing, plus local grid refinement
- **Gridgen quadtree** (`mesh-generation-gridgen`) - a quadtree unstructured (DISV) grid built with Gridgen *(this notebook)*
- **Triangle & Voronoi** (`mesh-generation-triangle-voronoi`) - unstructured (DISV) grids built with Triangle and Voronoi

## Imports and setup

Import FloPy and the helpers, define the watershed extent and contour levels,
load the fine topographic raster, and read the shared boundary/river geometry.

In [ ]:
%matplotlib inline
import json
import pathlib as pl

import flopy
import flopy.plot.styles as styles
import matplotlib.pyplot as plt
import numpy as np
from notebook_helpers import set_idomain, string2geom
from shapely.geometry import LineString

In [ ]:
# problem extent and display settings
Lx, Ly = 180000, 100000
extent = (0, Lx, 0, Ly)
vmin, vmax = 0.0, 100.0
levels = np.arange(10, 110, 10)

# the fine topography every grid samples, and the shared watershed geometry
data_path = pl.Path("../data/mesh-generation")
fine_topo = flopy.utils.Raster.load(data_path / "fine_topo.asc")
geometries = json.loads((data_path / "watershed_geometry.json").read_text())

boundary_polygon = string2geom(geometries["boundary"])
bp = np.array(boundary_polygon)
sgs = [string2geom(geometries[f"streamseg{i}"]) for i in range(1, 5)]

Two small helpers used for every grid below: `resample_topo` samples the fine
topographic raster onto a grid, and `river_intersection` marks the cells each
river segment crosses.

In [ ]:
def resample_topo(grid):
    """Sample the fine topography raster onto a model grid."""
    return fine_topo.resample_to_grid(
        grid, band=fine_topo.bands[0], method="linear", extrapolate_edges=True
    )


def river_intersection(grid, all_intersections=False):
    """Return an array flagged 1 in cells the river segments cross."""
    ixs = flopy.utils.GridIntersect(grid)
    cellids = []
    for sg in sgs:
        kw = {}
        if all_intersections:
            kw = dict(return_all_intersections=True)
        v = ixs.intersect(LineString(sg), sort_by_cellid=True, **kw)
        cellids += v["cellids"].tolist()
    arr = np.zeros(grid.shape[1:])
    for loc in cellids:
        arr[loc] = 1
    return arr


def draw_boundary_river(ax, river_alpha=1.0):
    ax.plot(bp[:, 0], bp[:, 1], "k-")
    for sg in sgs:
        sa = np.array(sg)
        ax.plot(sa[:, 0], sa[:, 1], "b-", alpha=river_alpha)

## Build the quadtree grid

Start from a coarse (5 km) base grid, then add Gridgen refinement features: a polygon around the area of interest and the river lines. Gridgen writes to a workspace under `models/`.

In [ ]:
from flopy.utils.gridgen import Gridgen

gridgen_ws = pl.Path("models/mesh-generation-gridgen")

# area of interest to refine (the LGR child region from the rectilinear notebook)
refine_poly = [
    [
        (65000.0, 40000.0),
        (65000.0, 60000.0),
        (90000.0, 60000.0),
        (90000.0, 40000.0),
        (65000.0, 40000.0),
    ]
]

# a coarse base grid for Gridgen to refine
sim = flopy.mf6.MFSimulation()
gwf = flopy.mf6.ModflowGwf(sim)
dx = dy = 5000.0
flopy.mf6.ModflowGwfdis(gwf, nrow=int(Ly / dy), ncol=int(Lx / dx), delr=dy, delc=dx)

gridgen_ws.mkdir(parents=True, exist_ok=True)
g = Gridgen(gwf.modelgrid, model_ws=gridgen_ws)
g.add_refinement_features([refine_poly], "polygon", 2, range(1))
g.add_refinement_features(sgs, "line", 2, range(1))
g.build(verbose=False)

quadtree_grid = flopy.discretization.VertexGrid(**g.get_gridprops_vertexgrid())
set_idomain(quadtree_grid, boundary_polygon)

top_qg = resample_topo(quadtree_grid)
intersection_qg = river_intersection(quadtree_grid)

In [ ]:
with styles.USGSMap():
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.set_aspect("equal")
    pmv = flopy.plot.PlotMapView(modelgrid=quadtree_grid, ax=ax)
    pmv.plot_array(top_qg, ec="0.75", vmin=vmin, vmax=vmax)
    pmv.plot_array(intersection_qg, masked_values=[0], alpha=0.2, cmap="Reds_r")
    pmv.contour_array(top_qg, levels=levels, linewidths=0.3, colors="white")
    pmv.plot_inactive(zorder=100)
    draw_boundary_river(ax)
    ax.set_title("Gridgen quadtree grid")

**Recap.** Gridgen turns a coarse base grid into a quadtree DISV grid, quartering cells near the area of interest and the river while leaving the rest of the domain coarse. The Triangle and Voronoi notebook builds fully unstructured grids over the same watershed.